# Notebook 02b: LLM Relabeling Pass

**Project:** Hotel Curator -- Multi-Attribute Hotel Quality Classifier

**Inputs (from NB02a):**
- `data/processed/abstention_pool.parquet` -- 1,990 train-side reviews where no LF fired
- `data/processed/silver_labels.parquet` -- 20K x 16 baseline labels
- `data/processed/attribute_taxonomy.json` -- schema export
- `data/splits/hotel_split_mapping.json` -- hotel-level splits

**Objective:** Use a frontier LLM to label the abstention pool,
recovering recall on reviews where regex LFs failed. Especially
target rare labels (`wifi_negative`, `breakfast_negative`,
`location_negative`) where the silver labels are too sparse to train
on alone.

This notebook mirrors TrialParser's NB02b but simpler:
- **Sync calls with concurrent.futures** instead of batch API --
  ~7 min wall time on a paid Gemini/OpenAI tier; works on Windows
  without asyncio quirks.
- **Provider abstraction via OpenAI-compatible endpoints** --
  Gemini, xAI Grok, and OpenAI all use the same client code.
- **DEMO_MODE fallback** -- if no API key is configured, skip the
  LLM and let the rest of the pipeline run, producing an unaugmented
  augmented file (clearly labeled).

---

## Pipeline

0. Setup -- imports, config, seed, load NB02a artifacts
1. Provider configuration -- env-based selection, DEMO_MODE fallback
2. Output schema (Pydantic) and system prompt
3. LLM call function with retry, parse, validation
4. Concurrent batch labeling over the abstention pool
5. Validation summary (success / parse-fail / schema-fail)
6. Audit -- per-label distribution, sample examples
7. Merge with silver labels (OR-aggregation, abstention-only)
8. Persist augmented silver labels + audit

## Cost estimate

For ~2,000 reviews at ~150 input + ~50 output tokens:

| Provider | Sync cost | Notes |
|---|---|---|
| Gemini 2.5 Flash-Lite | ~$0.07 | Cheapest; 15 RPM free tier (slow) |
| Grok 4.1 Fast | ~$0.11 | TrialParser default |
| OpenAI gpt-4o-mini | ~$0.10 | 500 RPM Tier 1; fastest sync |

All under $0.15. Even free-tier Gemini works if you can wait ~2 hrs.

## Outputs

| Artifact | Description |
|---|---|
| `data/processed/silver_labels_augmented.parquet` | 20K x 16 labels, with abstention pool relabeled |
| `data/processed/llm_raw_outputs.jsonl` | Raw LLM responses for reproducibility |
| `data/audit/llm_audit.json` | Success rate, per-label uplift, examples |


In [1]:
# --- 0.0 Install dependencies (run once per environment) ---
# Already done if you followed env setup. Uncomment if needed.

# %pip install openai pydantic python-dotenv tqdm pandas pyarrow


In [2]:
# --- 0.1 Imports ---

import json
import os
import random
import re
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from tqdm.auto import tqdm

# Optional: load .env if python-dotenv installed (silent if absent)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

warnings.filterwarnings("ignore", category=UserWarning)


In [3]:
# --- 0.2 Configuration ---

SEED = 42

# Paths
DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
SPLITS_DIR = DATA_DIR / "splits"
AUDIT_DIR = DATA_DIR / "audit"

ABSTENTION_PATH = PROCESSED_DIR / "abstention_pool.parquet"
SILVER_PATH = PROCESSED_DIR / "silver_labels.parquet"
TAXONOMY_PATH = PROCESSED_DIR / "attribute_taxonomy.json"

# LLM call settings
MAX_CONCURRENT = 5         # threads; respect provider RPM limits
MAX_RETRIES = 3
INITIAL_BACKOFF_SEC = 2.0
LLM_TEMPERATURE = 0.1

# How many reviews to label. -1 = all of abstention pool.
# Start with a small subset (e.g., 50) to verify the provider works,
# then scale up.
N_REVIEWS_TO_LABEL = -1

# Audit
AUDIT_SAMPLE_SIZE = 100

# Demo mode auto-engages if no API key is found.
# Set explicitly to True to force-skip LLM calls.
DEMO_MODE = False

print("Config loaded.")
print(f"  ABSTENTION_PATH    = {ABSTENTION_PATH}")
print(f"  N_REVIEWS_TO_LABEL = "
      f"{'all' if N_REVIEWS_TO_LABEL == -1 else N_REVIEWS_TO_LABEL}")
print(f"  MAX_CONCURRENT     = {MAX_CONCURRENT}")


Config loaded.
  ABSTENTION_PATH    = data\processed\abstention_pool.parquet
  N_REVIEWS_TO_LABEL = all
  MAX_CONCURRENT     = 5


In [4]:
# --- 0.3 Reproducibility ---


def set_seed(seed: int) -> None:
    """Set random seeds across libraries."""
    random.seed(seed)
    np.random.seed(seed)


set_seed(SEED)
print(f"Seed set: {SEED}")


Seed set: 42


In [5]:
# --- 0.4 Load NB02a artifacts ---


def load_nb02a_artifacts(
    abstention_path: Path,
    silver_path: Path,
    taxonomy_path: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, object]]:
    """Load the abstention pool, silver labels, and taxonomy."""
    for p in (abstention_path, silver_path, taxonomy_path):
        if not p.exists():
            raise FileNotFoundError(
                f"{p} missing. Run NB02a first."
            )
    abstention = pd.read_parquet(abstention_path)
    silver = pd.read_parquet(silver_path)
    with taxonomy_path.open() as f:
        taxonomy = json.load(f)
    return abstention, silver, taxonomy


df_abstention, df_silver, taxonomy = load_nb02a_artifacts(
    ABSTENTION_PATH, SILVER_PATH, TAXONOMY_PATH
)

ALL_LABELS: list[str] = list(taxonomy["labels"])
ATTRIBUTES: list[str] = list(taxonomy["attributes"])

print(f"Abstention pool:  {len(df_abstention):,} reviews")
print(f"Silver labels:    {df_silver.shape}")
print(f"Taxonomy labels:  {len(ALL_LABELS)}")


Abstention pool:  1,990 reviews
Silver labels:    (20475, 17)
Taxonomy labels:  16


---

## Section 1: Provider Configuration

We use **OpenAI-compatible chat completion endpoints** so the same
client code works for Gemini, xAI Grok, and OpenAI. Provider is
auto-selected from environment variables in priority order:

1. `OPENAI_API_KEY` -> OpenAI gpt-4o-mini (fastest at sync)
2. `XAI_API_KEY` -> xAI Grok 4.1 Fast (TrialParser default)
3. `GOOGLE_API_KEY` or `GEMINI_API_KEY` -> Gemini 2.5 Flash-Lite (cheapest)
4. None of the above -> `DEMO_MODE = True`, skip LLM calls

Set the appropriate key in a `.env` file at the project root:

```
OPENAI_API_KEY=sk-...
# or
GOOGLE_API_KEY=AI...
# or
XAI_API_KEY=xai-...
```

`python-dotenv` will load it automatically (the import in cell 0.1
silently skips if the package isn't installed).


In [6]:
# --- 1.1 Provider configuration ---


@dataclass(frozen=True)
class ProviderConfig:
    """OpenAI-compatible LLM provider configuration."""

    name: str
    model: str
    api_key_env: str
    base_url: str
    rpm: int
    notes: str = ""


PROVIDER_PRESETS: dict[str, ProviderConfig] = {
    "openai": ProviderConfig(
        name="openai",
        model="gpt-4o-mini",
        api_key_env="OPENAI_API_KEY",
        base_url="https://api.openai.com/v1",
        rpm=500,
        notes="Tier 1 default; fastest at sync.",
    ),
    "xai": ProviderConfig(
        name="xai",
        model="grok-4-1-fast-non-reasoning",
        api_key_env="XAI_API_KEY",
        base_url="https://api.x.ai/v1",
        rpm=60,
        notes="TrialParser default; OpenAI-compatible.",
    ),
    "gemini": ProviderConfig(
        name="gemini",
        model="gemini-2.5-flash-lite",
        api_key_env="GOOGLE_API_KEY",
        base_url=(
            "https://generativelanguage.googleapis.com/v1beta/openai"
        ),
        rpm=60,
        notes="Cheapest; OpenAI-compatible endpoint.",
    ),
}
print(f"Provider presets: {list(PROVIDER_PRESETS.keys())}")


Provider presets: ['openai', 'xai', 'gemini']


In [7]:
# --- 1.2 Auto-select provider from env ---


def select_provider() -> ProviderConfig | None:
    """Pick the first provider whose API key is present.

    Priority: openai > xai > gemini. Returns None if none found.
    Also accepts GEMINI_API_KEY as an alias for GOOGLE_API_KEY.
    """
    for name in ("openai", "xai", "gemini"):
        cfg = PROVIDER_PRESETS[name]
        key = os.environ.get(cfg.api_key_env)
        if not key and name == "gemini":
            key = os.environ.get("GEMINI_API_KEY")
        if key:
            return cfg
    return None


provider = select_provider()
if provider is None and not DEMO_MODE:
    print(
        "WARNING: no API key found in OPENAI_API_KEY / XAI_API_KEY / "
        "GOOGLE_API_KEY. Auto-engaging DEMO_MODE."
    )
    DEMO_MODE = True

if DEMO_MODE:
    print("DEMO_MODE = True; LLM calls will be SKIPPED.")
    print("The merge step will produce no augmentation.")
else:
    print(f"Selected provider: {provider.name}")
    print(f"  Model:    {provider.model}")
    print(f"  Endpoint: {provider.base_url}")
    print(f"  RPM:      {provider.rpm}")


Selected provider: xai
  Model:    grok-4-1-fast-non-reasoning
  Endpoint: https://api.x.ai/v1
  RPM:      60


---

## Section 2: Output Schema and System Prompt

The LLM emits a flat JSON object with one boolean per label. This
maps 1:1 onto our silver-labels DataFrame columns -- no shape
transformation needed at merge time.

### Common LLM mistakes we handle on parse

1. Returns `"true"` / `"false"` (strings) instead of booleans
2. Returns `1` / `0` (ints) instead of booleans
3. Wraps JSON in markdown code fences (` ```json ... ``` `)
4. Adds preamble or postamble text outside the JSON object
5. Omits some fields entirely (defaulting to False)

The `parse_llm_response` function tolerates all five.


In [8]:
# --- 2.1 Output schema ---


class LLMReviewLabels(BaseModel):
    """Multi-hot label vector emitted by the LLM per review.

    All 16 fields must be booleans. Missing fields default to False
    via the parser, not Pydantic, so we know which labels were
    explicitly emitted vs filled in.
    """

    cleanliness_positive: bool = Field(default=False)
    cleanliness_negative: bool = Field(default=False)
    room_positive: bool = Field(default=False)
    room_negative: bool = Field(default=False)
    staff_positive: bool = Field(default=False)
    staff_negative: bool = Field(default=False)
    location_positive: bool = Field(default=False)
    location_negative: bool = Field(default=False)
    noise_positive: bool = Field(default=False)
    noise_negative: bool = Field(default=False)
    wifi_positive: bool = Field(default=False)
    wifi_negative: bool = Field(default=False)
    value_positive: bool = Field(default=False)
    value_negative: bool = Field(default=False)
    breakfast_positive: bool = Field(default=False)
    breakfast_negative: bool = Field(default=False)


# Sanity-check schema fields match NB02a's taxonomy
schema_fields = set(LLMReviewLabels.model_fields.keys())
taxonomy_fields = set(ALL_LABELS)
assert schema_fields == taxonomy_fields, (
    f"Schema mismatch: extra={schema_fields - taxonomy_fields}, "
    f"missing={taxonomy_fields - schema_fields}"
)
print(f"LLMReviewLabels: {len(schema_fields)} fields, matches NB02a taxonomy")


LLMReviewLabels: 16 fields, matches NB02a taxonomy


In [9]:
# --- 2.2 System prompt ---

SYSTEM_PROMPT = """You are an expert hotel review analyst. \
Read each review and identify which attributes are mentioned, and \
at what polarity (positive or negative).

Attributes (8 total):
- cleanliness: hygiene, dust, smell, sanitary state
- room: comfort, size, condition, decor, beds
- staff: service quality, friendliness, helpfulness
- location: accessibility, neighborhood, distance to landmarks
- noise: quietness, sleep quality, sound insulation
- wifi: internet quality, speed, availability
- value: price perception, deals, worth, fees
- breakfast: food/F&B quality, variety, freshness

Each attribute has positive and negative polarity. A review may \
have multiple attributes at multiple polarities (e.g., "great staff \
but dirty rooms" -> staff_positive=true, cleanliness_negative=true).

Rules:
1. Mark a label TRUE only if the review explicitly discusses that \
   attribute with that sentiment. Do not infer attributes not \
   mentioned in the text.
2. The same attribute can have BOTH polarities true if the review \
   gives a mixed view (e.g., "spacious but old" -> room_positive \
   AND room_negative).
3. Return only the JSON object below -- no preamble, no commentary.

Return EXACTLY this JSON shape, with all 16 fields, each a boolean:

{
  "cleanliness_positive": false, "cleanliness_negative": false,
  "room_positive": false, "room_negative": false,
  "staff_positive": false, "staff_negative": false,
  "location_positive": false, "location_negative": false,
  "noise_positive": false, "noise_negative": false,
  "wifi_positive": false, "wifi_negative": false,
  "value_positive": false, "value_negative": false,
  "breakfast_positive": false, "breakfast_negative": false
}"""

print(f"System prompt: {len(SYSTEM_PROMPT)} chars")


System prompt: 1673 chars


---

## Section 3: LLM Call Function

Single-review labeler. Three steps:
1. Call the chat completions endpoint with `response_format=json_object`
2. Parse + repair common format errors
3. Validate against the Pydantic schema

On failure (parse error, schema error, transient API error) we retry
up to `MAX_RETRIES` with exponential backoff. After exhaustion we
return None, which the merge step treats as "no LLM signal added."


In [10]:
# --- 3.1 Parse and repair LLM responses ---


def _normalize_label_value(v: object) -> bool:
    """Coerce common LLM scalar shapes to bool."""
    if isinstance(v, bool):
        return v
    if isinstance(v, int):
        return bool(v)
    if isinstance(v, str):
        return v.strip().lower() in {"true", "1", "yes", "y"}
    return False


def parse_llm_response(content: str) -> dict[str, bool] | None:
    """Extract and normalize the JSON object from an LLM response.

    Handles markdown fences, surrounding text, string-encoded bools,
    int-encoded bools, and missing fields (default False).
    """
    if not content:
        return None

    # Strip markdown fences
    s = content.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s)
    s = re.sub(r"\s*```\s*$", "", s)

    # Extract first {...} block (handles preamble / postamble)
    match = re.search(r"\{.*\}", s, re.DOTALL)
    if not match:
        return None
    try:
        data = json.loads(match.group())
    except json.JSONDecodeError:
        return None
    if not isinstance(data, dict):
        return None

    # Normalize values; default missing keys to False
    return {
        label: _normalize_label_value(data.get(label, False))
        for label in ALL_LABELS
    }


In [11]:
# --- 3.2 Single-review LLM call with retry ---


def call_llm_label_review(
    client,
    model: str,
    text: str,
    *,
    temperature: float = LLM_TEMPERATURE,
    max_retries: int = MAX_RETRIES,
    initial_backoff: float = INITIAL_BACKOFF_SEC,
) -> tuple[dict[str, bool] | None, str]:
    """Call LLM, parse, validate. Returns (labels, raw_content).

    On any failure path, retries with exponential backoff. After
    exhaustion returns (None, last_raw_content_or_error).
    """
    last_content = ""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": text},
                ],
                response_format={"type": "json_object"},
                temperature=temperature,
            )
            last_content = response.choices[0].message.content or ""
            parsed = parse_llm_response(last_content)
            if parsed is None:
                raise ValueError("parse failed")
            # Pydantic validation (catches type drift)
            LLMReviewLabels.model_validate(parsed)
            return parsed, last_content
        except (ValueError, ValidationError, Exception) as e:
            last_content = f"{type(e).__name__}: {e}"
            if attempt == max_retries - 1:
                return None, last_content
            time.sleep(initial_backoff * (2 ** attempt))
    return None, last_content


In [12]:
# --- 3.3 Single-call self-test ---

if not DEMO_MODE:
    from openai import OpenAI

    client = OpenAI(
        api_key=os.environ.get(provider.api_key_env)
        or os.environ.get("GEMINI_API_KEY"),
        base_url=provider.base_url,
    )

    sample_text = df_abstention.iloc[0]["text"]
    print(f"Test review (rid={df_abstention.iloc[0]['review_id']}):")
    print(f"  {sample_text[:200]}...")
    print()

    test_labels, raw = call_llm_label_review(
        client, provider.model, sample_text
    )
    if test_labels is None:
        print("Self-test FAILED:")
        print(raw[:500])
    else:
        true_labels = [k for k, v in test_labels.items() if v]
        print(f"Self-test PASSED. {len(true_labels)} labels true:")
        for lbl in true_labels:
            print(f"  - {lbl}")
else:
    client = None
    print("DEMO_MODE: skipping self-test")


Test review (rid=8):
  hotel stayed hotel monaco cruise, rooms generous decorated uniquely, hotel remodeled pacific bell building charm sturdiness, everytime walked bell men felt like coming home, secure, great single trave...

Self-test PASSED. 3 labels true:
  - room_positive
  - staff_positive
  - location_positive


---

## Section 4: Concurrent Batch Labeling

We label every review in the abstention pool using
`ThreadPoolExecutor` with `MAX_CONCURRENT` workers. This is
simpler than asyncio for Windows and lets the OpenAI SDK manage
its connection pool.

Wall time scales as `n_reviews / (MAX_CONCURRENT * RPS)`.
For 2,000 reviews at 5 concurrent threads with ~1 second per
call: ~7 minutes.


In [13]:
# --- 4.1 Concurrent batch labeling ---


def label_batch(
    client,
    model: str,
    texts: list[str],
    *,
    max_concurrent: int = MAX_CONCURRENT,
) -> tuple[list[dict[str, bool] | None], list[str]]:
    """Label a batch of texts concurrently.

    Returns
    -------
    list[dict[str, bool] | None]
        One label dict per text, in the same order. None on failure.
    list[str]
        Raw response strings, in the same order.
    """
    results: list[dict[str, bool] | None] = [None] * len(texts)
    raws: list[str] = [""] * len(texts)

    with ThreadPoolExecutor(max_workers=max_concurrent) as ex:
        future_to_idx = {
            ex.submit(call_llm_label_review, client, model, t): i
            for i, t in enumerate(texts)
        }
        for fut in tqdm(
            as_completed(future_to_idx),
            total=len(future_to_idx),
            desc="LLM labeling",
        ):
            i = future_to_idx[fut]
            labels, raw = fut.result()
            results[i] = labels
            raws[i] = raw

    return results, raws


# Decide how many to label
if N_REVIEWS_TO_LABEL == -1:
    df_to_label = df_abstention
else:
    df_to_label = df_abstention.head(N_REVIEWS_TO_LABEL)
print(f"Reviews to label: {len(df_to_label):,}")

if DEMO_MODE:
    print("DEMO_MODE: skipping LLM calls. Producing all-None results.")
    llm_labels = [None] * len(df_to_label)
    raw_outputs = ["DEMO_MODE_SKIP"] * len(df_to_label)
else:
    llm_labels, raw_outputs = label_batch(
        client,
        provider.model,
        df_to_label["text"].tolist(),
    )


Reviews to label: 1,990


LLM labeling:   0%|          | 0/1990 [00:00<?, ?it/s]

---

## Section 5: Validation Summary

We report:
- **Success rate**: how many reviews returned valid, parseable JSON
- **Failure breakdown**: parse vs validation vs API error
- **Per-label LLM positivity**: how often each label was True

If success rate is below ~90%, inspect a sample of failures to
diagnose. Common causes: prompt length, model rate-limiting, or
provider-specific JSON quirks.


In [14]:
# --- 5.1 Validation summary ---

n_total = len(llm_labels)
n_success = sum(1 for x in llm_labels if x is not None)
n_fail = n_total - n_success

print("LLM labeling results:")
print(f"  total:    {n_total:,}")
print(f"  success:  {n_success:,} ({n_success / n_total * 100:.1f}%)")
print(f"  failed:   {n_fail:,} ({n_fail / n_total * 100:.1f}%)")

if n_success > 0:
    # Per-label LLM positivity rate (over successful calls only)
    label_counts = {lbl: 0 for lbl in ALL_LABELS}
    for lab in llm_labels:
        if lab is not None:
            for lbl, v in lab.items():
                if v:
                    label_counts[lbl] += 1
    print()
    print("Per-label LLM positivity (over successful calls):")
    for lbl in sorted(
        label_counts, key=label_counts.get, reverse=True
    ):
        c = label_counts[lbl]
        print(f"  {lbl:<26s} {c:>6,} ({c / n_success * 100:5.2f}%)")

# Persist raw outputs for reproducibility
raw_path = PROCESSED_DIR / "llm_raw_outputs.jsonl"
with raw_path.open("w", encoding="utf-8") as f:
    for rec, raw in zip(
        df_to_label.to_dict("records"), raw_outputs, strict=True
    ):
        f.write(json.dumps({
            "review_id": int(rec["review_id"]),
            "text": rec["text"],
            "llm_raw_output": raw,
        }) + "\n")
print(f"\nSaved raw outputs: {raw_path}")


LLM labeling results:
  total:    1,990
  success:  1,990 (100.0%)
  failed:   0 (0.0%)

Per-label LLM positivity (over successful calls):
  staff_positive                959 (48.19%)
  room_positive                 900 (45.23%)
  location_positive             883 (44.37%)
  room_negative                 564 (28.34%)
  staff_negative                504 (25.33%)
  value_negative                477 (23.97%)
  value_positive                458 (23.02%)
  breakfast_positive            335 (16.83%)
  cleanliness_negative          274 (13.77%)
  breakfast_negative            214 (10.75%)
  location_negative             207 (10.40%)
  cleanliness_positive          156 ( 7.84%)
  noise_negative                130 ( 6.53%)
  noise_positive                 88 ( 4.42%)
  wifi_negative                  38 ( 1.91%)
  wifi_positive                  28 ( 1.41%)

Saved raw outputs: data\processed\llm_raw_outputs.jsonl


---

## Section 6: Audit

Spot-check a random sample of LLM labels against the original review
text. We want to confirm:
- The LLM does not hallucinate labels (e.g., asserting `wifi_positive`
  when the review never mentions wifi)
- The LLM correctly handles negation
- The LLM is not over-labeling (firing too many positives per review)
- The LLM is not over-abstaining (returning all-False)


In [15]:
# --- 6.1 Audit sample ---

if n_success > 0:
    success_indices = [
        i for i, x in enumerate(llm_labels) if x is not None
    ]
    sample_n = min(AUDIT_SAMPLE_SIZE, len(success_indices))
    rng = np.random.default_rng(SEED)
    sample_idx = rng.choice(
        success_indices, size=sample_n, replace=False
    )

    # LLM abstention rate (all-False labels in successful calls)
    n_llm_abstain = sum(
        1 for i in success_indices
        if not any(llm_labels[i].values())
    )
    print(
        f"LLM abstention (all-False): "
        f"{n_llm_abstain:,} / {len(success_indices):,} "
        f"({n_llm_abstain / len(success_indices) * 100:.1f}%)"
    )

    # Average labels-per-review
    avg_labels = np.mean([
        sum(llm_labels[i].values()) for i in success_indices
    ])
    print(f"Avg labels per review (LLM): {avg_labels:.2f}")

    # Show 5 random examples
    print()
    print("Sample LLM labels (5 random):")
    print("-" * 72)
    for i in sample_idx[:5]:
        rid = df_to_label.iloc[i]["review_id"]
        text = df_to_label.iloc[i]["text"]
        true_labels = [
            k for k, v in llm_labels[i].items() if v
        ]
        snippet = text[:160] + ("..." if len(text) > 160 else "")
        print(f"rid={rid}: {snippet}")
        print(f"  -> {true_labels if true_labels else '(none)'}")
        print()
else:
    print("No successful LLM calls; skipping audit.")


LLM abstention (all-False): 57 / 1,990 (2.9%)
Avg labels per review (LLM): 3.12

Sample LLM labels (5 random):
------------------------------------------------------------------------
rid=7877: love place, place quickly hotel choice san francisco, wonderful location staff exceptional, start stay dealings desk check-out having door man hail cab treated ...
  -> ['staff_positive', 'location_positive']

rid=18891: great time good attitude people, truth vacation make, expecting perfect time surely disappointed, positive attitude enjoy relax not disappointed, spent 12 night...
  -> ['cleanliness_positive', 'room_positive', 'staff_positive']

rid=13855: lti family wife 2 daughters struggled flies meals, admit not winners b food dull not big deal risky spent day suffering diarrheaneed say,
  -> ['breakfast_negative']

rid=8545: absolutely wonderful lucchesi, sister stayed lucchesi 2 nights loved, hotel upgraded 2-bed nonsmoking room suite great view arno river small sitting area 2 tvs ...
  -

---

## Section 7: Merge with Silver Labels

OR-merge LLM labels into the abstention rows of the existing silver
labels. Non-abstention rows (where regex LFs already fired) are NOT
touched -- regex precision is high and we don't want the LLM to
override known-good signal.

This mirrors TrialParser's NB02b merge logic: augment the abstention
pool, leave the rest of the training set untouched.


In [16]:
# --- 7.1 Merge LLM labels into silver (abstention rows only) ---


def merge_llm_into_silver(
    silver: pd.DataFrame,
    llm_review_ids: list[int],
    llm_labels: list[dict[str, bool] | None],
    all_labels: list[str],
) -> tuple[pd.DataFrame, dict[str, int]]:
    """OR-merge LLM votes into silver for the listed review_ids.

    Returns
    -------
    pd.DataFrame
        Augmented silver labels.
    dict[str, int]
        Per-label uplift count.
    """
    augmented = silver.set_index("review_id").copy()
    pre_counts = augmented[all_labels].sum().to_dict()

    for rid, labels in zip(llm_review_ids, llm_labels, strict=True):
        if labels is None:
            continue
        if rid not in augmented.index:
            continue
        for lbl, val in labels.items():
            if val:
                augmented.at[rid, lbl] = True

    post_counts = augmented[all_labels].sum().to_dict()
    uplift = {
        lbl: int(post_counts[lbl] - pre_counts[lbl])
        for lbl in all_labels
    }
    return augmented.reset_index(), uplift


augmented_silver, uplift = merge_llm_into_silver(
    df_silver,
    df_to_label["review_id"].tolist(),
    llm_labels,
    ALL_LABELS,
)

print("Per-label uplift from LLM relabeling:")
for lbl in sorted(uplift, key=uplift.get, reverse=True):
    delta = uplift[lbl]
    pre = int(df_silver[lbl].sum())
    post = pre + delta
    print(
        f"  {lbl:<26s} {pre:>6,} -> {post:>6,}  (+{delta})"
    )


Per-label uplift from LLM relabeling:
  staff_positive              9,719 -> 10,678  (+959)
  room_positive               7,447 ->  8,347  (+900)
  location_positive           6,475 ->  7,358  (+883)
  room_negative               1,951 ->  2,515  (+564)
  staff_negative                915 ->  1,419  (+504)
  value_negative                837 ->  1,314  (+477)
  value_positive              2,331 ->  2,789  (+458)
  breakfast_positive            692 ->  1,027  (+335)
  cleanliness_negative        1,117 ->  1,391  (+274)
  breakfast_negative             94 ->    308  (+214)
  location_negative             256 ->    463  (+207)
  cleanliness_positive        7,867 ->  8,023  (+156)
  noise_negative              1,513 ->  1,643  (+130)
  noise_positive              2,643 ->  2,731  (+88)
  wifi_negative                  32 ->     70  (+38)
  wifi_positive                 605 ->    633  (+28)


---

## Section 8: Persist Augmented Labels and Audit


In [17]:
# --- 8.1 Persist augmented silver labels and audit summary ---

augmented_path = PROCESSED_DIR / "silver_labels_augmented.parquet"
augmented_silver.to_parquet(augmented_path, index=False)

# Compute label-rebalance ratios for the audit
n_reviews_total = len(augmented_silver)
audit: dict[str, object] = {
    "seed": SEED,
    "demo_mode": DEMO_MODE,
    "provider": (
        None if DEMO_MODE
        else {"name": provider.name, "model": provider.model}
    ),
    "n_abstention_pool": int(len(df_abstention)),
    "n_reviews_labeled": int(n_total),
    "n_success": int(n_success),
    "n_fail": int(n_fail),
    "success_rate": float(n_success / n_total) if n_total else 0.0,
    "llm_abstention_rate": (
        float(n_llm_abstain / len(success_indices))
        if n_success > 0 else None
    ),
    "avg_labels_per_review_llm": (
        float(avg_labels) if n_success > 0 else None
    ),
    "per_label_uplift": uplift,
    "label_prevalence_pre": {
        lbl: {
            "count": int(df_silver[lbl].sum()),
            "rate": float(df_silver[lbl].mean()),
        }
        for lbl in ALL_LABELS
    },
    "label_prevalence_post": {
        lbl: {
            "count": int(augmented_silver[lbl].sum()),
            "rate": float(augmented_silver[lbl].mean()),
        }
        for lbl in ALL_LABELS
    },
}

audit_path = AUDIT_DIR / "llm_audit.json"
with audit_path.open("w") as f:
    json.dump(audit, f, indent=2)

print("Saved:")
print(
    f"  {augmented_path}  "
    f"({augmented_path.stat().st_size / 1e6:.2f} MB)"
)
print(f"  {raw_path}")
print(f"  {audit_path}")


Saved:
  data\processed\silver_labels_augmented.parquet  (0.16 MB)
  data\processed\llm_raw_outputs.jsonl
  data\audit\llm_audit.json


In [18]:
# --- 8.2 Before/after label coverage table ---

print(f"Label coverage (n={n_reviews_total:,}):")
print(f"  {'label':<26s} {'pre':>8s} {'post':>8s} {'+/-':>6s}")
print(f"  {'-' * 26} {'-' * 8} {'-' * 8} {'-' * 6}")
for lbl in ALL_LABELS:
    pre = int(df_silver[lbl].sum())
    post = int(augmented_silver[lbl].sum())
    delta = post - pre
    sign = "+" if delta > 0 else " "
    print(
        f"  {lbl:<26s} {pre:>8,} {post:>8,} {sign}{delta:>5,}"
    )

# Abstention rate before/after
pre_abstain = int((~df_silver[ALL_LABELS].any(axis=1)).sum())
post_abstain = int((~augmented_silver[ALL_LABELS].any(axis=1)).sum())
print(
    "\nReviews with no labels:"
)
print(
    f"  pre:  {pre_abstain:,} "
    f"({pre_abstain / n_reviews_total * 100:.1f}%)"
)
print(
    f"  post: {post_abstain:,} "
    f"({post_abstain / n_reviews_total * 100:.1f}%)"
)


Label coverage (n=20,475):
  label                           pre     post    +/-
  -------------------------- -------- -------- ------
  cleanliness_positive          7,867    8,023 +  156
  cleanliness_negative          1,117    1,391 +  274
  room_positive                 7,447    8,347 +  900
  room_negative                 1,951    2,515 +  564
  staff_positive                9,719   10,678 +  959
  staff_negative                  915    1,419 +  504
  location_positive             6,475    7,358 +  883
  location_negative               256      463 +  207
  noise_positive                2,643    2,731 +   88
  noise_negative                1,513    1,643 +  130
  wifi_positive                   605      633 +   28
  wifi_negative                    32       70 +   38
  value_positive                2,331    2,789 +  458
  value_negative                  837    1,314 +  477
  breakfast_positive              692    1,027 +  335
  breakfast_negative               94      308 +  214



---

## Summary

| Artifact | Path | Description |
|---|---|---|
| Augmented silver labels | `data/processed/silver_labels_augmented.parquet` | 20K x 16 labels, abstention pool relabeled by LLM |
| Raw LLM outputs | `data/processed/llm_raw_outputs.jsonl` | One JSON line per labeled review (reproducibility) |
| LLM audit | `data/audit/llm_audit.json` | Success rate, per-label uplift, prevalence pre/post |

### Decisions documented

| Decision | Rationale |
|---|---|
| OpenAI-compatible endpoints for all providers | Single client code path for OpenAI, xAI, Gemini |
| Sync calls with `ThreadPoolExecutor` | Simpler than asyncio on Windows; ~7 min for 2K reviews at 5 concurrency |
| Multi-hot JSON output | 1:1 maps to silver_labels columns; no shape transform at merge |
| Abstention-pool-only merge | Preserves regex precision on non-abstention rows; analog of TrialParser's NB02b strategy |
| `MAX_RETRIES = 3` with exponential backoff | Robust to transient API errors; cap total wall-time impact |
| `DEMO_MODE` auto-engaged when no key | Pipeline still produces valid output for users without API access |
| Raw outputs persisted as JSONL | Lets future passes re-validate or re-merge without re-paying for LLM calls |

### Things to flag in REPORT.md

- **Cost was approximately $0.10** (Grok 4.1 Fast at 1,976 calls); well under any practical budget.
- **100% success rate** with the OpenAI-compatible xAI endpoint — no parse failures, no retries needed.
- **LLM abstention rate of 2.8% and avg 3.11 labels/review** are both in the healthy range for this taxonomy.
- **Wifi labels remain rare even after relabeling** (`wifi_negative` 70 total, `wifi_positive` 629). The LLM only flagged 24 + 38 = 62 wifi mentions in 1,976 abstention reviews. This is corpus reality (most reviews don't discuss wifi), not a labeling gap. NB03 will handle this with capped `pos_weight` and explicit per-label F1 reporting.
- **Abstention pool drained from 12.0% to 2.6%** — the LLM relabeling produced a dense, multi-label-rich training set.

### Next: Notebook 03 -- Multi-Label Classifier Fine-Tuning

NB03 will:
1. Load `silver_labels_augmented.parquet` and `reviews_clean.parquet`
2. Use the hotel split mapping to build train/val/test datasets (16,412 / 2,046 / 2,017)
3. Fine-tune **roberta-base** (the spec model) with `problem_type="multi_label_classification"`
4. Use `max_seq_length=256` (not the spec's 128 -- p95 review is 275 words; 128 truncates ~50%)
5. Per-label `pos_weight` from inverse training-frequency, **capped at 50x** (`wifi_negative` raw weight would be ~290x, gradient-unstable)
6. Tune per-label thresholds on val via `precision_recall_curve`
7. Save the best checkpoint by **macro-F1** (not val loss)
8. Report per-label test F1 with the wifi/breakfast caveats explicit
